In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
torch.manual_seed(42)
np.random.seed(42)
df = pd.read_csv('spiral_dataset.csv')
df

In [ ]:
df.info()

In [ ]:
df['label'].value_counts()

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(df['x1'], df['x2'], c=df['label'], cmap='viridis', s=10)
plt.show()

In [ ]:
X = df[['x1', 'x2']]
y = df['label']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
X_train, X_val, X_test = [torch.FloatTensor(a.values) for a in (X_train, X_val, X_test)]
y_train, y_val, y_test = [torch.LongTensor(a.values) for a in (y_train, y_val, y_test)]
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

In [ ]:
class Spiral(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Dropout(p=0.1),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(p=0.1),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(p=0.1),
            nn.Linear(64, 3)
        )

    def forward(self, x):
        return self.model(x)

model = Spiral()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=15)

In [ ]:
n_epochs = 600
patience = 60
best_val_loss = float('inf')
best_state = None
wait = 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
for epoch in range(n_epochs):
    model.train()
    batch_losses = []
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
    model.eval()
    with torch.no_grad():
        val_out = model(X_val)
        val_loss = criterion(val_out, y_val).item()
        val_acc = (val_out.argmax(1) == y_val).float().mean().item()
    scheduler.step(val_loss)
    history['train_loss'].append(sum(batch_losses) / len(batch_losses))
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            break
    if (epoch + 1) % 25 == 0:
        print(f"Epoch {epoch+1}/{n_epochs} | Loss: {history['train_loss'][-1]:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")
model.load_state_dict(best_state)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_xlabel('epoch')
axes[0].legend()
axes[1].plot(history['val_acc'])
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('val accuracy')
plt.tight_layout()
plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    test_pred = model(X_test).argmax(1)
test_acc = (test_pred == y_test).float().mean().item()
from sklearn.metrics import classification_report
print(classification_report(y_test, test_pred))

In [ ]:
xx, yy = np.meshgrid(np.linspace(-1.2, 1.2, 300), np.linspace(-1.2, 1.2, 300))
grid = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])
with torch.no_grad():
    zz = model(grid).argmax(1).numpy().reshape(xx.shape)
plt.figure(figsize=(7, 7))
plt.contourf(xx, yy, zz, alpha=0.3, cmap='viridis')
plt.scatter(df['x1'], df['x2'], c=df['label'], cmap='viridis', s=8)
plt.show()

In [ ]:
print(f'test accuracy: {test_acc:.2f}')